# SetFit HTS Two-Stage Classifier — Chapter → Heading

Two models trained sequentially:
1. **Chapter model** — predicts 2-digit chapter (65 classes, ~17k samples) → should hit 85-95%
2. **Heading model** — predicts 4-digit heading with chapter hint in the text (filtered to headings with ≥10 samples) → much easier task

At inference: predict chapter → prepend `"[ChXX] "` to text → predict heading.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or **L4 GPU** (Pro) → Save
2. Upload from `src/data/`:
   - `crossRulings-train.json`
   - `crossRulings-validation.json`
   - `crossRulings-test.json`

**Expected time:** ~60 min on T4, ~40 min on L4

**Why two-stage?** The flat heading model got 53% on 498+ headings. Most headings have <5 training samples.
Two-stage fixes this: chapter prediction is easy (65 classes), and heading-within-chapter is much smaller (avg 8 headings per chapter).

**If training already finished and you just need results / download:** Run **Step 8** through **Step 12** only (same session, runtime still connected). Do *not* re-run Steps 1–7 or you'll lose the models in memory. The zip is created in Step 12.

**If you see `NameError: name 'heading_model' is not defined`:** Runtime restarted — models are gone. Either: **(A)** Run **all** cells (Runtime → Run all) to retrain (~40–60 min), or **(B)** If you have **setfit-two-stage.zip** from a previous run, use **Resume from zip** below to skip retraining: run Steps 1–4, then the Resume cell, then Steps 8–12.

## Step 1: Install Dependencies

In [ ]:
# Same pinned versions that worked in the previous notebook
!pip install -q "setfit==1.1.3" "transformers>=4.44,<5.0" "sentence-transformers>=3.0,<4.0" torch scikit-learn datasets

import setfit, transformers
print(f"setfit=={setfit.__version__}, transformers=={transformers.__version__}")

## Step 2: Upload Training Data

In [ ]:
from google.colab import files
import json
import os

print("Upload all 3 files from src/data/:")
print("  crossRulings-train.json")
print("  crossRulings-validation.json")
print("  crossRulings-test.json")
print()

uploaded = files.upload()

for f in ['crossRulings-train.json', 'crossRulings-validation.json', 'crossRulings-test.json']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {f} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {f}")

## Step 3: Load & Analyze Data

In [ ]:
import json
import os
from collections import Counter
from datasets import Dataset
import gc

def load_rulings(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    with open(filepath, 'r') as f:
        data = json.load(f)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and 'rulings' in data:
            return data['rulings']
        else:
            raise ValueError(f'Unexpected format in {filepath}')

def extract_hts(ruling):
    """Extract clean HTS code from a ruling dict."""
    text = ruling.get('productDescription') or ruling.get('product_description', '')
    codes = ruling.get('htsCodes') or ruling.get('hts_codes', [])
    if not text or not codes:
        return None, None, None, None
    code = codes[0] if isinstance(codes, list) else codes
    code = code.replace('.', '').replace(' ', '').strip()
    if len(code) < 4:
        return None, None, None, None
    return text.strip(), code[:2], code[:4], code[:6]

print("Loading data...")
train_raw = load_rulings('crossRulings-train.json')
val_raw = load_rulings('crossRulings-validation.json')
test_raw = load_rulings('crossRulings-test.json')
print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")

# Analyze heading distribution to decide min_samples filter
heading_counts = Counter()
chapter_counts = Counter()
for r in train_raw:
    text, ch, hd, _ = extract_hts(r)
    if ch:
        chapter_counts[ch] += 1
        heading_counts[hd] += 1

print(f"\nChapters: {len(chapter_counts)} unique")
print(f"Headings: {len(heading_counts)} unique")
print(f"  >=10 samples: {sum(1 for c in heading_counts.values() if c >= 10)} headings")
print(f"  <10 samples:  {sum(1 for c in heading_counts.values() if c < 10)} headings (will use AI fallback)")

## Step 4: Prepare Datasets for Both Stages

- **Chapter dataset**: all samples, label = 2-digit chapter
- **Heading dataset**: only headings with ≥10 training samples, text prefixed with `[ChXX]`

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────
MIN_HEADING_SAMPLES = 10   # Drop headings with fewer training samples
MAX_TRAIN_SAMPLES = 12000  # OOM safety cap (reduce to 8000 for T4 if needed)

# Headings that pass the filter
valid_headings = {h for h, c in heading_counts.items() if c >= MIN_HEADING_SAMPLES}
print(f"Headings passing filter (>={MIN_HEADING_SAMPLES} samples): {len(valid_headings)}")
print(f"Headings dropped: {len(heading_counts) - len(valid_headings)}")

def build_chapter_dataset(rulings, max_samples=None):
    """All samples, label = 2-digit chapter."""
    texts, labels = [], []
    for i, r in enumerate(rulings):
        if max_samples and i >= max_samples:
            break
        text, ch, hd, _ = extract_hts(r)
        if text and ch:
            texts.append(text)
            labels.append(ch)
    return Dataset.from_dict({'text': texts, 'label': labels})

def build_heading_dataset(rulings, valid_headings_set, max_samples=None):
    """Only valid headings, text prefixed with [ChXX] for chapter hint."""
    texts, labels = [], []
    skipped = 0
    for i, r in enumerate(rulings):
        if max_samples and i >= max_samples:
            break
        text, ch, hd, _ = extract_hts(r)
        if not text or not hd:
            continue
        if hd not in valid_headings_set:
            skipped += 1
            continue
        # Prefix with chapter hint — this is what inference will do too
        texts.append(f"[Ch{ch}] {text}")
        labels.append(hd)
    if skipped > 0:
        print(f"  Skipped {skipped} samples from rare headings")
    return Dataset.from_dict({'text': texts, 'label': labels})

# Build chapter datasets
print("\n--- Chapter datasets ---")
ch_train = build_chapter_dataset(train_raw, max_samples=MAX_TRAIN_SAMPLES)
ch_val = build_chapter_dataset(val_raw)
ch_test = build_chapter_dataset(test_raw)
print(f"Train: {len(ch_train)} | Val: {len(ch_val)} | Test: {len(ch_test)}")
print(f"Unique chapters: {len(set(ch_train['label']))}")

# Build heading datasets (filtered + chapter-prefixed)
print("\n--- Heading datasets ---")
hd_train = build_heading_dataset(train_raw, valid_headings, max_samples=MAX_TRAIN_SAMPLES)
hd_val = build_heading_dataset(val_raw, valid_headings)
hd_test = build_heading_dataset(test_raw, valid_headings)
print(f"Train: {len(hd_train)} | Val: {len(hd_val)} | Test: {len(hd_test)}")
print(f"Unique headings: {len(set(hd_train['label']))}")

# Free raw data
del train_raw, val_raw, test_raw
gc.collect()

# Show a few heading examples to verify prefix
print("\nSample heading texts (with chapter prefix):")
for i in range(min(3, len(hd_train))):
    print(f"  {hd_train['label'][i]}: {hd_train['text'][i][:100]}...")

## Optional: Resume from zip (skip retraining)

**Only if** the runtime restarted and you have **setfit-two-stage.zip** from a previous run: run Steps 1–4 above, then run the cell below. Upload the zip when prompted, then run Steps 8–12. Skip Steps 5, 6, 7 (no retraining).

In [ ]:
# Run this only if you have setfit-two-stage.zip and want to skip retraining
from google.colab import files
from setfit import SetFitModel
import os

uploaded = files.upload()  # Select setfit-two-stage.zip
if 'setfit-two-stage.zip' not in uploaded:
    raise SystemExit("Upload setfit-two-stage.zip (from a previous run).")
!unzip -o setfit-two-stage.zip

CHAPTER_DIR = './setfit-hts-chapter'
HEADING_DIR = './setfit-hts-heading-v2'
chapter_model = SetFitModel.from_pretrained(CHAPTER_DIR)
heading_model = SetFitModel.from_pretrained(HEADING_DIR)
print("Models loaded. Run Steps 8–12 (evaluate, e2e, samples, save, download).")

## Step 5: Train Stage 1 — Chapter Classifier

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
import torch
import gc

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.cuda.empty_cache()
else:
    gpu_name = "CPU"
    gpu_mem = 0
    print("No GPU — this will be slow")

gc.collect()

# GPU-adaptive settings (same logic as working notebook)
if gpu_mem >= 40:
    batch_size = 16
    num_iterations = 15
elif gpu_mem >= 20:
    batch_size = 8
    num_iterations = 10
else:
    batch_size = 4
    num_iterations = 5

CHAPTER_DIR = './setfit-hts-chapter'

print(f"\nTraining CHAPTER model ({len(set(ch_train['label']))} classes)...")
print(f"Samples: {len(ch_train)} | Batch: {batch_size} | Iterations: {num_iterations} | Epochs: 2")

chapter_model = SetFitModel.from_pretrained(
    'sentence-transformers/all-MiniLM-L6-v2',
    labels=list(set(ch_train['label']))
)

ch_args = TrainingArguments(
    batch_size=batch_size,
    num_epochs=2,
    evaluation_strategy='no',
    save_strategy='epoch',
    output_dir=CHAPTER_DIR,
    logging_steps=50,
    num_iterations=num_iterations,
    report_to='none',
)

ch_trainer = Trainer(
    model=chapter_model,
    args=ch_args,
    train_dataset=ch_train,
)

ch_trainer.train()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nChapter model training complete!")

## Step 6: Evaluate Chapter Model

In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score

print("Evaluating chapter model...")
start = time.time()
ch_preds = chapter_model.predict(ch_test['text'])
ch_accuracy = accuracy_score(ch_test['label'], ch_preds)
ch_time = (time.time() - start) / len(ch_test) * 1000

# Top-3 chapter accuracy (convert all to native Python to avoid numpy.int64 key errors)
ch_top3 = None
try:
    X = chapter_model.encode(ch_test['text'])
    proba = chapter_model.model_head.predict_proba(X)
    labels = [str(c) for c in chapter_model.model_head.classes_]
    top3_idx = [[int(k) for k in row] for row in np.argsort(proba, axis=1)[:, -3:]]
    true_labels = [str(x) for x in ch_test['label']]
    ch_top3 = sum(
        1 for i, true in enumerate(true_labels)
        if true in [labels[j] for j in top3_idx[i]]
    ) / len(true_labels)
except Exception as e:
    print(f"(Top-3 skipped: {e})")

print(f"\n{'='*60}")
print(f"CHAPTER MODEL")
print(f"  Top-1 Accuracy: {ch_accuracy*100:.1f}%")
if ch_top3 is not None:
    print(f"  Top-3 Accuracy: {ch_top3*100:.1f}%")
print(f"  Inference: {ch_time:.2f}ms/sample")
print(f"{'='*60}")

## Step 7: Train Stage 2 — Heading Classifier (with chapter prefix)

In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

HEADING_DIR = './setfit-hts-heading-v2'

print(f"\nTraining HEADING model ({len(set(hd_train['label']))} classes, chapter-prefixed)...")
print(f"Samples: {len(hd_train)} | Batch: {batch_size} | Iterations: {num_iterations} | Epochs: 3")

heading_model = SetFitModel.from_pretrained(
    'sentence-transformers/all-MiniLM-L6-v2',
    labels=list(set(hd_train['label']))
)

hd_args = TrainingArguments(
    batch_size=batch_size,
    num_epochs=3,          # Heading needs more epochs than chapter
    evaluation_strategy='no',
    save_strategy='epoch',
    output_dir=HEADING_DIR,
    logging_steps=50,
    num_iterations=num_iterations,
    report_to='none',
)

hd_trainer = Trainer(
    model=heading_model,
    args=hd_args,
    train_dataset=hd_train,
)

hd_trainer.train()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nHeading model training complete!")

## Step 8: Evaluate Heading Model

In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score

print("Evaluating heading model (on chapter-prefixed test set)...")
start = time.time()
hd_preds = heading_model.predict(hd_test['text'])
hd_accuracy = accuracy_score(hd_test['label'], hd_preds)
hd_time = (time.time() - start) / len(hd_test) * 1000

# Top-3 heading accuracy (convert all to native Python to avoid numpy.int64 key errors)
hd_top3 = None
try:
    X = heading_model.encode(hd_test['text'])
    proba = heading_model.model_head.predict_proba(X)
    labels = [str(c) for c in heading_model.model_head.classes_]
    top3_idx = [[int(k) for k in row] for row in np.argsort(proba, axis=1)[:, -3:]]
    true_labels = [str(x) for x in hd_test['label']]
    hd_top3 = sum(
        1 for i, true in enumerate(true_labels)
        if true in [labels[j] for j in top3_idx[i]]
    ) / len(true_labels)
except Exception as e:
    print(f"(Top-3 skipped: {e})")

print(f"\n{'='*60}")
print(f"HEADING MODEL (chapter-prefixed, filtered)")
print(f"  Top-1 Accuracy: {hd_accuracy*100:.1f}%")
if hd_top3 is not None:
    print(f"  Top-3 Accuracy: {hd_top3*100:.1f}%")
print(f"  Inference: {hd_time:.2f}ms/sample")
print(f"{'='*60}")

## Step 9: End-to-End Two-Stage Evaluation

Simulates real inference: chapter model predicts chapter → prefix text → heading model predicts heading.

In [ ]:
# Reload raw test data for end-to-end eval (need unprefixed text + true heading)
test_raw = load_rulings('crossRulings-test.json')

e2e_texts = []
e2e_true_chapters = []
e2e_true_headings = []

for r in test_raw:
    text, ch, hd, _ = extract_hts(r)
    if text and ch and hd:
        e2e_texts.append(text)
        e2e_true_chapters.append(ch)
        e2e_true_headings.append(hd)

del test_raw

print(f"End-to-end test samples: {len(e2e_texts)}")
print(f"(includes all headings, even rare ones not in heading model)\n")

# Stage 1: predict chapter
start = time.time()
pred_chapters = chapter_model.predict(e2e_texts)

# Stage 2: prefix with predicted chapter, predict heading
prefixed_texts = [f"[Ch{ch}] {text}" for ch, text in zip(pred_chapters, e2e_texts)]
pred_headings = heading_model.predict(prefixed_texts)
e2e_time = (time.time() - start) / len(e2e_texts) * 1000

# Accuracy metrics
ch_correct = sum(1 for p, t in zip(pred_chapters, e2e_true_chapters) if p == t)
hd_correct = sum(1 for p, t in zip(pred_headings, e2e_true_headings) if p == t)

# Heading accuracy GIVEN correct chapter (shows heading model quality in isolation)
ch_match_indices = [i for i, (p, t) in enumerate(zip(pred_chapters, e2e_true_chapters)) if p == t]
hd_given_ch = sum(1 for i in ch_match_indices if pred_headings[i] == e2e_true_headings[i])

print(f"{'='*60}")
print(f"END-TO-END TWO-STAGE RESULTS")
print(f"  Chapter accuracy:  {ch_correct}/{len(e2e_texts)} = {ch_correct/len(e2e_texts)*100:.1f}%")
print(f"  Heading accuracy:  {hd_correct}/{len(e2e_texts)} = {hd_correct/len(e2e_texts)*100:.1f}%")
if ch_match_indices:
    print(f"  Heading|correct chapter: {hd_given_ch}/{len(ch_match_indices)} = {hd_given_ch/len(ch_match_indices)*100:.1f}%")
print(f"  Avg inference: {e2e_time:.2f}ms/sample (both models)")
print(f"{'='*60}")

# Compare to old flat model
print(f"\nOld flat heading model: 52.4% (498 classes, no chapter hint)")
print(f"New two-stage heading:  {hd_correct/len(e2e_texts)*100:.1f}% (filtered + chapter prefix)")

## Step 10: Test Sample Predictions

In [ ]:
test_examples = [
    "cotton t-shirt",
    "plastic water bottle",
    "leather wallet",
    "stainless steel kitchen knife",
    "wooden dining table",
    "lithium-ion battery pack for electric vehicles",
    "women's knit cardigan 60% cotton 40% polyester",
    "hydraulic excavator",
]

print("Two-stage predictions:\n")
for text in test_examples:
    ch = chapter_model.predict([text])[0]
    hd = heading_model.predict([f'[Ch{ch}] {text}'])[0]
    print(f"  '{text}'")
    print(f"    Chapter {ch} -> Heading {hd}")
    print()

## Step 11: Save Metrics & Models

In [ ]:
import os

# Save chapter model
print(f"Saving chapter model to {CHAPTER_DIR}...")
chapter_model.save_pretrained(CHAPTER_DIR)

ch_metrics = {
    'accuracy': ch_accuracy,
    'top3_accuracy': ch_top3,
    'total_samples': len(ch_test),
    'inference_time_ms': ch_time,
    'training_samples': len(ch_train),
    'model_name': 'setfit-hts-chapter',
    'training_level': 'chapter',
}
with open(f'{CHAPTER_DIR}/metrics.json', 'w') as f:
    json.dump(ch_metrics, f, indent=2)

# Save heading model
print(f"Saving heading model to {HEADING_DIR}...")
heading_model.save_pretrained(HEADING_DIR)

hd_metrics = {
    'accuracy': hd_accuracy,
    'top3_accuracy': hd_top3,
    'total_samples': len(hd_test),
    'inference_time_ms': hd_time,
    'training_samples': len(hd_train),
    'model_name': 'setfit-hts-heading-v2',
    'training_level': 'heading',
    'chapter_prefixed': True,
    'min_heading_samples': MIN_HEADING_SAMPLES,
    'valid_headings': len(valid_headings),
    'e2e_accuracy': hd_correct / len(e2e_texts) if e2e_texts else None,
}
with open(f'{HEADING_DIR}/metrics.json', 'w') as f:
    json.dump(hd_metrics, f, indent=2)

# Save the valid headings list (needed at inference to know which headings the model covers)
with open(f'{HEADING_DIR}/valid_headings.json', 'w') as f:
    json.dump(sorted(list(valid_headings)), f)

print("\nModels and metrics saved!")

## Step 12: Download Models

In [ ]:
# Zip both models together
!zip -r setfit-two-stage.zip ./setfit-hts-chapter/ ./setfit-hts-heading-v2/

print("\nDownloading setfit-two-stage.zip...")
files.download('setfit-two-stage.zip')

print(f"\nDone! Extract and start the inference server with both models:")
print(f"  python scripts/setfit-inference-server.py --chapter-model models/setfit-hts-chapter --heading-model models/setfit-hts-heading-v2")